## 0. Install library

In [5]:
!pip -q install transformers datasets seqeval accelerate scikit-learn

## 1. Upload file export Label Studio
Pilih file `project-...json` hasil **Export → JSON** dari Label Studio.

In [6]:
from google.colab import files
uploaded = files.upload()
EXPORT_PATH = list(uploaded.keys())[0]
print("File:", EXPORT_PATH)

Saving project-2-at-2026-11-23-01-40-4dc0ac79.json to project-2-at-2026-11-23-01-40-4dc0ac79.json
File: project-2-at-2026-11-23-01-40-4dc0ac79.json


## 2. Ubah anotasi Label Studio menjadi format BIO

Label Studio menyimpan span sebagai posisi karakter (`start`, `end`). Model butuh **satu tag per kata**:
- `B-LABEL` : kata pertama dalam frasa
- `I-LABEL` : kata lanjutan dalam frasa yang sama
- `O` : kata di luar frasa aspek

Catatan teknis: posisi di export Label Studio dihitung per karakter (code point), sama seperti Python.
Karena itu tiap span **dicek ulang**: teks di posisi itu harus sama dengan teks frasa yang tersimpan.
Kalau tidak sama (misalnya highlight bergeser di review yang ada emoji), posisinya diperbaiki otomatis
dengan mencari frasa yang sama di dekat posisi semula.

In [7]:
import json, re, collections

def fix_span(text, start, end, span_text):
    # Pastikan posisi span cocok dengan teks frasanya. Kalau bergeser, cari frasa itu di dekatnya.
    if text[start:end] == span_text:
        return start, end, False
    cands = [m.start() for m in re.finditer(re.escape(span_text), text)]
    if not cands:
        return None
    s0 = min(cands, key=lambda c: abs(c - start))
    return s0, s0 + len(span_text), True

raw = json.load(open(EXPORT_PATH, encoding="utf-8"))

examples, skipped, repaired, dropped = [], 0, 0, 0
for task in raw:
    anns = [a for a in task.get("annotations", []) if not a.get("was_cancelled")]
    if not anns:
        skipped += 1
        continue
    ann = anns[-1]                                   # pakai anotasi terakhir
    text = task["data"]["text"]

    # kata = potongan teks yang dipisah spasi, lengkap dengan posisinya
    words = [(m.group(), m.start(), m.end()) for m in re.finditer(r"\S+", text)]
    tags = ["O"] * len(words)

    spans = []
    for r in ann["result"]:
        if r.get("type") != "labels":
            continue
        v = r["value"]
        fixed = fix_span(text, v["start"], v["end"], v["text"])
        if fixed is None:
            dropped += 1
            continue
        s0, e0, was_fixed = fixed
        repaired += was_fixed
        spans.append((s0, e0, v["labels"][0]))

    for s, e, label in sorted(spans):
        hit = [i for i, (_, ws, we) in enumerate(words) if ws < e and we > s]   # kata yang tersentuh span
        for k, i in enumerate(hit):
            if tags[i] == "O":
                tags[i] = ("B-" if k == 0 else "I-") + label

    examples.append({
        "tokens": [w for w, _, _ in words],
        "tags": tags,
        "product_id": task["data"].get("product_id", -1),
        "product_type": task["data"].get("product_type", "?"),
    })

print(f"Review terpakai: {len(examples)} | dilewati (belum dianotasi): {skipped}")
print(f"Span diperbaiki posisinya: {repaired} | span dibuang (frasa tidak ketemu): {dropped}")
print("\nContoh:")
for e in examples[:2]:
    print(list(zip(e["tokens"], e["tags"])))

Review terpakai: 274 | dilewati (belum dianotasi): 0
Span diperbaiki posisinya: 1 | span dibuang (frasa tidak ketemu): 0

Contoh:
[('Proses', 'B-PERFORMA'), ('cepet', 'I-PERFORMA'), ('tp', 'O'), ('pengiriman', 'B-PENGIRIMAN_LAYANAN'), ('yg', 'I-PENGIRIMAN_LAYANAN'), ('lama.', 'I-PENGIRIMAN_LAYANAN'), ('iPhone', 'B-KEASLIAN_KESESUAIAN'), ('terdeteksi', 'I-KEASLIAN_KESESUAIAN'), ('ORI.', 'I-KEASLIAN_KESESUAIAN'), ('Dpt', 'O'), ('unit', 'O'), ('GDN', 'O'), ('atau', 'O'), ('******', 'O'), ('jd', 'O'), ('dpt', 'B-PENGIRIMAN_LAYANAN'), ('proteksi', 'I-PENGIRIMAN_LAYANAN'), ('gratis', 'I-PENGIRIMAN_LAYANAN'), ('selama', 'I-PENGIRIMAN_LAYANAN'), ('12', 'I-PENGIRIMAN_LAYANAN'), ('bulan.', 'I-PENGIRIMAN_LAYANAN'), ('Jgn', 'O'), ('ragu2', 'O'), ('beli', 'O'), ('di', 'O'), ('sini', 'O'), ('karena', 'O'), ('di', 'O'), ('jamin', 'O'), ('ORI', 'O')]
[('kualitas', 'B-KUALITAS_FISIK'), ('tv', 'I-KUALITAS_FISIK'), ('sangat', 'I-KUALITAS_FISIK'), ('bagus.', 'I-KUALITAS_FISIK'), ('sudah', 'O'), ('pakai', 

## 3. Daftar label
Label diambil otomatis dari data, jadi kalau kamu menambah atau menghapus label di Label Studio, notebook tetap jalan.

In [8]:
entity_types = sorted({t[2:] for e in examples for t in e["tags"] if t != "O"})
labels = ["O"] + [f"{p}-{t}" for t in entity_types for p in ("B", "I")]
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

counts = collections.Counter(t[2:] for e in examples for t in e["tags"] if t.startswith("B-"))
print("Jumlah frasa per label (seluruh data):")
for k, v in counts.most_common():
    print(f"  {k:22s}{v}")

Jumlah frasa per label (seluruh data):
  PENGIRIMAN_LAYANAN    348
  KEASLIAN_KESESUAIAN   131
  KUALITAS_FISIK        127
  PERFORMA              79
  BATERAI_DAYA          54
  HARGA                 39
  LAYAR_TAMPILAN        33
  SUARA                 22
  KONEKSI               9
  KEMUDAHAN             7


## 4. Split train / validation / test **per produk**
Review dari produk yang sama **tidak boleh** muncul di dua split sekaligus.
Kalau tidak, model bisa "menghafal" produk dan angka F1 jadi terlalu bagus (kebocoran data).

In [9]:
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
groups = [e["product_id"] for e in examples]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
trainval_idx, test_idx = next(gss.split(examples, groups=groups))

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.18, random_state=SEED)
tr_rel, va_rel = next(gss2.split(trainval_idx, groups=[groups[i] for i in trainval_idx]))

train = [examples[trainval_idx[i]] for i in tr_rel]
val   = [examples[trainval_idx[i]] for i in va_rel]
test  = [examples[i] for i in test_idx]

print(f"train={len(train)}  val={len(val)}  test={len(test)}")
for name, split in [("train", train), ("val", val), ("test", test)]:
    c = collections.Counter(t[2:] for e in split for t in e["tags"] if t.startswith("B-"))
    print(f"\n{name}:", dict(c.most_common()))
print("\nProduk di split yang beda tidak boleh sama:",
      not (set(e['product_id'] for e in train) & set(e['product_id'] for e in test)))

train=186  val=40  test=48

train: {'PENGIRIMAN_LAYANAN': 248, 'KEASLIAN_KESESUAIAN': 93, 'KUALITAS_FISIK': 89, 'PERFORMA': 49, 'BATERAI_DAYA': 35, 'HARGA': 30, 'LAYAR_TAMPILAN': 19, 'SUARA': 12, 'KONEKSI': 7, 'KEMUDAHAN': 5}

val: {'PENGIRIMAN_LAYANAN': 42, 'KUALITAS_FISIK': 17, 'KEASLIAN_KESESUAIAN': 17, 'PERFORMA': 12, 'BATERAI_DAYA': 11, 'LAYAR_TAMPILAN': 8, 'SUARA': 7, 'HARGA': 7, 'KONEKSI': 1, 'KEMUDAHAN': 1}

test: {'PENGIRIMAN_LAYANAN': 58, 'KEASLIAN_KESESUAIAN': 21, 'KUALITAS_FISIK': 21, 'PERFORMA': 18, 'BATERAI_DAYA': 8, 'LAYAR_TAMPILAN': 6, 'SUARA': 3, 'HARGA': 2, 'KONEKSI': 1, 'KEMUDAHAN': 1}

Produk di split yang beda tidak boleh sama: True


## 5. Tokenisasi dan penyelarasan label
IndoBERT memecah satu kata menjadi beberapa *subword*. Label hanya dipasang di **subword pertama** tiap kata.
Sisanya diberi `-100` supaya diabaikan saat menghitung loss.

In [10]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "indobenchmark/indobert-base-p1"
# Alternatif kalau mau coba model lain:
# MODEL_NAME = "indolem/indobert-base-uncased"
# MODEL_NAME = "cahya/bert-base-indonesian-522M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
assert tokenizer.is_fast, "Butuh fast tokenizer. Coba MODEL_NAME alternatif di atas."

def encode(batch):
    enc = tokenizer(batch["tokens"], is_split_into_words=True, truncation=True, max_length=128)
    all_labels = []
    for i, tags in enumerate(batch["tags"]):
        prev, lab = None, []
        for w in enc.word_ids(batch_index=i):
            if w is None:
                lab.append(-100)                       # token khusus [CLS]/[SEP]
            elif w != prev:
                lab.append(label2id[tags[w]])          # subword pertama dari kata
            else:
                lab.append(-100)                       # subword lanjutan
            prev = w
        all_labels.append(lab)
    enc["labels"] = all_labels
    return enc

def to_ds(split):
    ds = Dataset.from_list([{"tokens": e["tokens"], "tags": e["tags"]} for e in split])
    return ds.map(encode, batched=True, remove_columns=["tokens", "tags"])

train_ds, val_ds, test_ds = to_ds(train), to_ds(val), to_ds(test)
print(train_ds)

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/186 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 186
})


## 6. Model dan training
Data kita kecil (sekitar 200 review untuk training), jadi epoch dibuat banyak dan model terbaik
(berdasarkan F1 di validation) yang disimpan.

In [11]:
import math
import numpy as np, torch
from transformers import (AutoConfig, AutoModelForTokenClassification, TrainingArguments, Trainer,
                          DataCollatorForTokenClassification, set_seed)
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

set_seed(SEED)

# Checkpoint IndoBERT membawa daftar label bawaan (5 label). Kita timpa dengan label kita sendiri.
config = AutoConfig.from_pretrained(MODEL_NAME)
config.id2label = id2label
config.label2id = label2id
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, config=config)

# Pengaman: jumlah output model harus sama dengan jumlah label kita
assert model.config.num_labels == len(labels), (model.config.num_labels, len(labels))
assert model.classifier.out_features == len(labels)
print("Jumlah label model:", model.config.num_labels)

def decode(logits, label_ids):
    pred = np.argmax(logits, axis=2)
    y_true, y_pred = [], []
    for p, l in zip(pred, label_ids):
        y_true.append([id2label[x] for x in l if x != -100])
        y_pred.append([id2label[a] for a, x in zip(p, l) if x != -100])
    return y_true, y_pred

def compute_metrics(eval_pred):
    logits, label_ids = eval_pred
    y_true, y_pred = decode(logits, label_ids)
    return {"precision": precision_score(y_true, y_pred),
            "recall": recall_score(y_true, y_pred),
            "f1": f1_score(y_true, y_pred)}

EPOCHS, BATCH = 20, 8
total_steps = math.ceil(len(train_ds) / BATCH) * EPOCHS
warmup = int(0.1 * total_steps)         # 10% langkah awal untuk pemanasan learning rate

args = TrainingArguments(
    output_dir="ner_out",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=warmup,
    eval_strategy="epoch",          # kalau error "unexpected keyword", ganti jadi evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    processing_class=tokenizer,     # kalau error "unexpected keyword", ganti jadi tokenizer=tokenizer
    compute_metrics=compute_metrics,
)
trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (39 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=480, training_loss=0.2791668256123861, metrics={'train_runtime': 2436.7301, 'train_samples_per_second': 1.527, 'train_steps_per_second': 0.197, 'total_flos': 102954314719800.0, 'train_loss': 0.2791668256123861, 'epoch': 20.0})

## 7. Evaluasi di test set
Angka yang dilaporkan di README adalah **F1 per label di test set**, bukan di validation.
Dengan data sekecil ini, angkanya akan naik-turun cukup besar. Itu wajar, tulis apa adanya.

In [12]:
pred = trainer.predict(test_ds)
y_true, y_pred = decode(pred.predictions, pred.label_ids)

print(f"F1 keseluruhan : {f1_score(y_true, y_pred):.3f}")
print(f"Precision      : {precision_score(y_true, y_pred):.3f}")
print(f"Recall         : {recall_score(y_true, y_pred):.3f}\n")
print(classification_report(y_true, y_pred, digits=3))

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


F1 keseluruhan : 0.613
Precision      : 0.576
Recall         : 0.655

                     precision    recall  f1-score   support

       BATERAI_DAYA      0.364     0.500     0.421         8
              HARGA      1.000     1.000     1.000         2
KEASLIAN_KESESUAIAN      0.478     0.524     0.500        21
          KEMUDAHAN      0.000     0.000     0.000         1
            KONEKSI      0.000     0.000     0.000         1
     KUALITAS_FISIK      0.500     0.571     0.533        21
     LAYAR_TAMPILAN      0.833     0.833     0.833         6
 PENGIRIMAN_LAYANAN      0.621     0.707     0.661        58
           PERFORMA      0.684     0.722     0.703        18
              SUARA      1.000     1.000     1.000         3

          micro avg      0.576     0.655     0.613       139
          macro avg      0.548     0.586     0.565       139
       weighted avg      0.588     0.655     0.619       139



## 8. Analisis error
Lihat kasus yang salah. Ini bahan paling berharga buat bagian *Error analysis* di README.

In [13]:
def bio_to_spans(tokens, tags):
    spans, cur = [], None
    for tok, tag in zip(tokens, tags):
        if tag.startswith("B-"):
            if cur: spans.append(cur)
            cur = [tag[2:], [tok]]
        elif tag.startswith("I-") and cur and cur[0] == tag[2:]:
            cur[1].append(tok)
        else:
            if cur: spans.append(cur)
            cur = None
    if cur: spans.append(cur)
    return [(lab, " ".join(ws)) for lab, ws in spans]

shown = 0
for ex, gold, pr in zip(test, y_true, y_pred):
    g, p = bio_to_spans(ex["tokens"], gold), bio_to_spans(ex["tokens"][:len(pr)], pr)
    if g != p:
        print("TEKS :", " ".join(ex["tokens"]))
        print("  gold:", g)
        print("  pred:", p, "\n")
        shown += 1
    if shown >= 8:
        break

TEKS : Proses cepet tp pengiriman yg lama. iPhone terdeteksi ORI. Dpt unit GDN atau ****** jd dpt proteksi gratis selama 12 bulan. Jgn ragu2 beli di sini karena di jamin ORI
  gold: [('PERFORMA', 'Proses cepet'), ('PENGIRIMAN_LAYANAN', 'pengiriman yg lama.'), ('KEASLIAN_KESESUAIAN', 'iPhone terdeteksi ORI.'), ('PENGIRIMAN_LAYANAN', 'dpt proteksi gratis selama 12 bulan.')]
  pred: [('PENGIRIMAN_LAYANAN', 'Proses cepet'), ('PENGIRIMAN_LAYANAN', 'pengiriman yg lama.'), ('KEASLIAN_KESESUAIAN', 'terdeteksi ORI.')] 

TEKS : mantap, adminnya fast respon. green sealed dan imei terdaftar.
  gold: [('PENGIRIMAN_LAYANAN', 'adminnya fast respon.'), ('KEASLIAN_KESESUAIAN', 'imei terdaftar.')]
  pred: [('PENGIRIMAN_LAYANAN', 'adminnya fast respon.'), ('KEASLIAN_KESESUAIAN', 'green sealed')] 

TEKS : barang ori official, fastcharging, quick respon seller
  gold: [('KEASLIAN_KESESUAIAN', 'barang ori official,'), ('BATERAI_DAYA', 'fastcharging,'), ('PENGIRIMAN_LAYANAN', 'quick respon seller')]
  pred: 

## 9. Coba model dengan kalimat baru
Fungsi `extract_aspects` ini juga yang nanti dipakai di chatbot untuk meringkas review.

In [14]:
def extract_aspects(text):
    words = text.split()
    enc = tokenizer(words, is_split_into_words=True, truncation=True, max_length=128, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**enc).logits[0]
    pred_ids = logits.argmax(-1).tolist()
    tags, prev = [], None
    for idx, w in enumerate(enc.word_ids()):
        if w is not None and w != prev:
            tags.append(id2label[pred_ids[idx]])
        prev = w
    return bio_to_spans(words[:len(tags)], tags)

for s in ["Baterai awet banget tapi layarnya agak redup, packing rapi dan pengiriman cepat",
          "Suara jernih, harga murah, barang original"]:
    print(s, "->", extract_aspects(s))

Baterai awet banget tapi layarnya agak redup, packing rapi dan pengiriman cepat -> [('BATERAI_DAYA', 'Baterai awet'), ('LAYAR_TAMPILAN', 'layarnya'), ('PENGIRIMAN_LAYANAN', 'packing rapi'), ('PENGIRIMAN_LAYANAN', 'pengiriman cepat')]
Suara jernih, harga murah, barang original -> [('SUARA', 'Suara'), ('HARGA', 'harga murah,'), ('KEASLIAN_KESESUAIAN', 'barang original')]


## 10. Simpan model
Model disimpan ke Google Drive (atau diunduh sebagai zip) supaya bisa dipakai di chatbot.

In [15]:
SAVE_DIR = "ner_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

!zip -qr ner_model.zip ner_model
from google.colab import files
files.download("ner_model.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>